In [2]:
import os
import sys

import json
import random
import time
import pandas as pd
import seaborn as sns

from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path

In [3]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

import corpus
import api

from dev import rel, print_epi_summary
from data_config import DATA_CONFIG, FEW_SHOT_PATH
from few_shot import get_few_shot_examples
from error_sampling import error_summary, sample_errors
from plotting import plot_dist_comparison

import pipeline

# Config.

### Development Config.

In [4]:
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at 2026-08-23 11:39


### EPI Config.

In [24]:
# ---------- Manual EPI config. entry ----------
EPI_NUM = "002"
DATASET_SPLIT = "train"

# ---------- Config. whether EPI config's API should be called again ----------
CALL_API = True

# ---------- Config. whether error distribution figure should be saved ----------
SAVE_FIG = True

# Corpus Loading

### Load Abstracts

In [25]:
# ---------- Get abstracts path ----------
ABSTRACTS_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["abstracts"]
SAMPLE_SIZE = DATA_CONFIG[DATASET_SPLIT]["sample_size"]

# ---------- Load corpus from path ----------
abstracts_corpus = corpus.load_corpus(ABSTRACTS_PATH, sample_size=SAMPLE_SIZE)
print(f"Abstracts dataset length: {len(abstracts_corpus)}")

Abstracts dataset length: 400


### Get Ground Truths

In [26]:
# ---------- Manually Set Ground Truth Paths ----------
GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["ground_truths"]
ENTITIES_GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["entities_ground_truths"]

# ---------- Fetch BioRED Ground Truths ----------
abstracts_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, GT_PATH)
entities_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, ENTITIES_GT_PATH)

In [27]:
entities_ground_truths.head()

,pmid,entity_id,text,entity_type,identifier,start,end,split
0,10491763,0,Hepatocyte nuclear factor-6,GeneOrGeneProduct,3175,0,27,train
1,10491763,1,type II diabetes,DiseaseOrPhenotypicFeature,D003924,74,90,train
2,10491763,2,insulin,GeneOrGeneProduct,3630,140,147,train
3,10491763,3,hepatocyte nuclear factor (HNF)-6,GeneOrGeneProduct,3175,184,217,train
4,10491763,4,maturity-onset diabetes,DiseaseOrPhenotypicFeature,D003924,292,315,train


In [28]:
abstracts_ground_truths.head()

,pmid,entity_1,entity_1_type,entity_1_identifier,relation,entity_2,entity_2_type,entity_2_identifier,split
0,10491763,Hepatocyte nuclear factor-6,GeneOrGeneProduct,3175,Association,type II diabetes,DiseaseOrPhenotypicFeature,D003924,train
1,10491763,glucose,ChemicalEntity,D005947,Positive_Correlation,insulin,GeneOrGeneProduct,3630,train
2,10491763,glucose,ChemicalEntity,D005947,Association,type II diabetes,DiseaseOrPhenotypicFeature,D003924,train
3,10661407,Langerin,GeneOrGeneProduct,50489,Bind,mannose,ChemicalEntity,D008358,train
4,10788334,breast or ovarian cancer,DiseaseOrPhenotypicFeature,D001943,Positive_Correlation,5382insC,SequenceVariant,c|INS|5382|C,train


In [29]:
entities_ground_truths["entity_type"].unique()

<StringArray>
[         'GeneOrGeneProduct', 'DiseaseOrPhenotypicFeature',
             'ChemicalEntity',              'OrganismTaxon',
            'SequenceVariant',                   'CellLine']
Length: 6, dtype: str

In [30]:
os.getcwd()

'e:\\Users\\wesle\\biomedical_kg_thesis\\src\\experiments'

In [31]:
# ---------- Turn ground truth DataFrame into structured dict ----------
ground_truths = corpus.get_gt_dict(abstracts_ground_truths, entities_ground_truths)

### Few-Shot Construction

In [32]:
# ---------- If FS block does not exist, create FS block, else pass ----------
if not os.path.exists(FEW_SHOT_PATH):

    few_shot_block = get_few_shot_examples(
        path_to_train_set=DATA_CONFIG["train"]["paths"]["abstracts"],
        path_to_train_gts=DATA_CONFIG["train"]["paths"]["ground_truths"],
        biored_train_samples=abstracts_corpus,
        few_shot_export_path=FEW_SHOT_PATH
    )
    print(f"Generated and exported new few-shot block to '{FEW_SHOT_PATH}'")
else:
    with open(FEW_SHOT_PATH) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{FEW_SHOT_PATH}' at {datetime.now().strftime('%Y-%m-%d %H:%M')}.")

Imported existing few-shot block from '../../data/few_shot/few_shot_block.txt' at 2026-08-23 13:12.


### Import BioRED Extraction Guidelines

In [33]:
# ---------- Import BioRED guidelines text file for prompt refinement ----------
with open("../../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

# ---------- Print preview ----------
print(f"{biored_ext_guidelines[:500]}...")

## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover...


# OpenAI Luna API Call

### EPI Setup

In [34]:
# ---------- Create EPI setup dictionary ----------
# - Keys: "dataset", "id", "eval_version", "notes", "reuse_api_call", "prompt"
epi_setup = pipeline.setup_epi(EPI_NUM, DATASET_SPLIT)

# ---------- Print summary ----------
print_epi_summary(
    epi_num=EPI_NUM,
    prompt_version=epi_setup["prompt"]["version"],
    eval_version=epi_setup["eval_version"],
    notes=epi_setup["notes"],
    reuse_api_call=epi_setup["reuse_api_call"]
)

Cell ran at 2026-08-23 13:12 for epi_002
 - Prompt version: v2
 - Evaluation version: v2
 - Notes: Implemented BioRED extraction guidelines and entity and relation type rules & added relaxed matching.
 - Reuse API Call: False


### API Call

In [ ]:
# ---------- API call / import existing EPI API log ----------
if CALL_API:
    # ---------- Create client ----------
    client = api.create_client(timeout=120, max_retries=2)

    # ---------- If prompt version is different from previous EPI, call API, else pass ----------
    if not epi_setup["reuse_api_call"]:

        print(f"Running API call for {epi_setup["id"]} ({epi_setup["dataset"]})...")

        # Fetch raw prompt template
        prompt_template = epi_setup["prompt"]["template"]

        # Begin timer
        start_time = time.perf_counter()

        # Initiate outputs list
        outputs = []

        current_abstract_num = 1

        # Begin looping through abstract dataset rows - one call per row
        for index, row in abstracts_corpus.iterrows():
            abstract = row["abstract"]

            num_abstracts = abstracts_corpus.shape[0]

            # Replace prompt template's placeholders with abstract, few_shot & guidelines
            prompt = (
                prompt_template
                .replace("{abstract}", abstract)
                .replace("{few_shot_block}", few_shot_block)
                .replace("{biored_ext_guidelines}", biored_ext_guidelines)
            )

            # Store row's response
            response = client.responses.create(
                model="gpt-5.6-luna",
                input=prompt
            )

            # Append response to outputs list
            outputs.append({
                "pmid": row["pmid"],
                "output": response.output_text
            })

            progress_time = time.perf_counter() - start_time
            avg_time_per_abstract = progress_time / current_abstract_num
            est_time_remaining = avg_time_per_abstract * (num_abstracts - current_abstract_num)
            minutes = int(est_time_remaining // 60)
            seconds = est_time_remaining % 60

            print(
                f"\rCompleteled extraction for abstract {current_abstract_num} of {num_abstracts} "
                f"({current_abstract_num / num_abstracts * 100:.1f}%; "
                f"Est. time remaining: {minutes}m {seconds:.0f}s).",
                end="",
                flush=True
            )
            current_abstract_num += 1

        # End time, store elapsed time & print result
        elapsed_seconds = time.perf_counter() - start_time
        print(f"\n\nAPI calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts\n")

        print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f" - Prompt verion: {epi_setup["prompt"]["version"]}")
        print(f" - Evalaution verion: {epi_setup["eval_version"]}")
        print(f" - Run notes: {epi_setup["notes"]}")
        print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
    else:
        prev_epi_id = f"epi_{int(EPI_NUM) - 1:03d}"

        with open(f"{DATA_CONFIG[DATASET_SPLIT]["paths"]["epi_dir"]}/{prev_epi_id}.json") as f:
            prev_epi_log = json.load(f)

        outputs = prev_epi_log["outputs"]
        prompt_template = prev_epi_log["prompt"]
        elapsed_seconds = prev_epi_log["time_taken"]

        print(f"Reused API call from {prev_epi_id}.")

    output_info = {
        "epi_id": epi_setup["id"],
        "outputs": outputs,
        "time_taken": elapsed_seconds,
        "raw_prompt": prompt_template,
        "epi_notes": epi_setup["notes"],
        "prompt_version": epi_setup["prompt"]["version"],
        "eval_version": epi_setup["eval_version"],
        "export_path": DATA_CONFIG[DATASET_SPLIT]["paths"]["epi_dir"]
    }
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (PIPELINE_RUN = False).")

Created client:
 - Timeout: 120
 - Max retries: 2

Running API call for epi_002 (biored_train)...
Completeled extraction for abstract 5 of 400 (1.2%; Est. time remaining: 121m 58s).

In [ ]:
# ---------- Parse outputs, evaluate extractions, export EPI results ----------
if CALL_API:
    epi_log = pipeline.process_epi(output_info, ground_truths, dataset=epi_setup["dataset"])
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} ('CALL_API = False').")

Parsed 400 extractions, 1 failed to parse as JSON
Saved epi_001 to ../../data/results/epis/biored_train


# Error Analysis

### Summary Table

In [ ]:
# ---------- If EPI API call does not exist in kernel state, import existing EPI log ----------
if not CALL_API:
    try:
        with open(f"../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json") as f:
            epi_log = json.load(f)
    except:
        raise ValueError(
            f"EPI log cannot be imported: "
            f"'../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json' does not exist."
        )

error_summary(epi_log)

`epi_001` Error Summary:


Metric,Precision,Recall,F1
Entity,0.0,0.0,0.0
Relation,0.0,0.0,0.0


### Distribution Comparison

In [ ]:
if EPI_NUM == "001":
    print(
        "Extraction distribution analaysis is not conducted on 'epi_001' - "
        "extractions are not bound to fixed sets."
    )
elif epi_setup["reuse_api_call"] == True:
    print(
        f"EPI uses previous EPI's prompt - "
        f"did not plot ground truth/extraction distribution figure for `epi_{EPI_NUM}`."
    )
else:
    plot_dist_comparison(epi_log, ground_truths, save=SAVE_FIG)


Extraction distribution analaysis is not conducted on 'epi_001' - extractions are not bound to fixed sets.


## Relations

### False Positives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False positives ----------
    relation_fp = epi_log["errors"]["errors"]["relations"]["false_positives"]

    print(f"Count: {len(relation_fp)}")

    display(sample_errors(relation_fp, sample_size=200))

Count: 11047
Sample size: 200


[[21070631, 'inflammation', 'occurs in', 'brain'],
 [16412238, 'y355x', 'has_zygosity', 'heterozygous mutation'],
 [29045486,
  'pro-apoptotic/inflammatory signaling',
  'contributes_to',
  'apoptotic cell death'],
 [19923525,
  'nimodipine (nimo)',
  'does not improve in absence of hypotension',
  'latency time'],
 [17615423,
  'rhabdomyolysis',
  'associated_with_elevated',
  'alanine aminotransferase'],
 [17083016, 'sp-a1', 'encodes', 'surfactant protein a (sp-a)'],
 [18310445, 'bmp-4', 'favours', 'fsh rather than lh synthesis and secretion'],
 [20801104,
  '-616c/g',
  'located_in_promoter_region_of',
  'dopamine d4 receptor (drd4) gene'],
 [20477932,
  'cocaine administration',
  'decreases',
  'glutathione peroxidase activity'],
 [24535067, 'fentanyl', 'prevents', 'etomidate-induced myoclonus'],
 [18366737, 'fh variants', 'includes', '93 pathogenic variants'],
 [15748645, 'ultraviolet (uv) b', 'increases_risk_of', 'skin cancer'],
 [26102294, 'failure of autophagy', 'causes accumu

### False Negatives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False negatives ----------
    relation_fn = epi_log["errors"]["errors"]["relations"]["false_negatives"]

    print(f"Count: {len(relation_fn)}")

    display(sample_errors(relation_fn, sample_size=200))

Count: 3834
Sample size: 200


[[25263533, 't-cell receptor', 'Negative_Correlation', 'bcl6'],
 [16786513, 'mcardle disease', 'Association', 'c.1471c>t'],
 [18541230,
  'puromycin aminonucleoside',
  'Positive_Correlation',
  'renal dysfunction'],
 [20046642, 'carmustine', 'Positive_Correlation', 'malondialdehyde'],
 [20583543, '46,xy dsd', 'Association', 'srd5a2'],
 [10788334, 'brca1', 'Association', 'brca1 abnormalities'],
 [16596970,
  'pilocarpine',
  'Positive_Correlation',
  'impairment in auditory location discrimination'],
 [19841052, 'crack cocaine', 'Positive_Correlation', 'hiv infection'],
 [17379047, 'valsartan', 'Comparison', 'hydrochlorothiazide'],
 [24971338, 'cyclosporine', 'Association', 'tp53'],
 [16786513, 'mcardle disease', 'Association', 'p.k543x'],
 [27084744, 'sox10', 'Association', 'cancer'],
 [24126708, 'levodopa', 'Negative_Correlation', "parkinson's disease"],
 [19218574, 'phosphatidylinositol 3-kinase', 'Association', 'glucose'],
 [21810259, 'ido', 'Association', 'hiv-1-infected'],
 [1761

## Entities

### False Positives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False positives ----------
    entities_fp = epi_log["errors"]["errors"]["entities"]["false_positives"]

    print(f"Count: {len(entities_fp)}")

    display(sample_errors(entities_fp))

Count: 10538
Sample size: 20


[[18410548, 'family history', 'clinical_factor'],
 [15266215, 'osteoarthritis', 'disease'],
 [16810074, 'venomotor tone', 'physiological process'],
 [26102294, 'persistent dna damage', 'biological process'],
 [16634859, 'nondrinking group', 'control group'],
 [20129423, 'liver', 'organ'],
 [16391785, 'trichostatin a', 'chemical compound'],
 [15749661, '1469t->g', 'genetic variant'],
 [26731607, 'p38 mitogen-activated protein kinase', 'protein'],
 [25622904, 'cellular senescence', 'biological process'],
 [18487244, 'catalytic domain', 'protein domain'],
 [17615423, 'alanine aminotransferase', 'laboratory_measurement'],
 [28472177, 'cell adhesion', 'biological process'],
 [24911645, 'animals', 'organism'],
 [16843501, 'endogenous oxidative dna lesions', 'DNA damage'],
 [27640183, 'prolonged epo exposure', 'experimental intervention'],
 [23791840, 'wild-type (wt) embryos', 'biological specimen'],
 [15188772, 'medication labeling', 'intervention'],
 [19306381, 'focal adhesion kinase', 'kin

### False Negatives

In [ ]:
if DATASET_SPLIT == "train":
    # ---------- False negatives ----------
    entities_fn = epi_log["errors"]["errors"]["entities"]["false_negatives"]

    print(f"Count: {len(entities_fn)}")

    display(sample_errors(entities_fn))

Count: 6087
Sample size: 20


[[16120104, '111g', 'SequenceVariant'],
 [8755918, 'cerebellar ataxia', 'DiseaseOrPhenotypicFeature'],
 [20528871, 'cyclophosphamide', 'ChemicalEntity'],
 [21135151, 'aids', 'DiseaseOrPhenotypicFeature'],
 [22104738, 'c.920t>g', 'SequenceVariant'],
 [15278670, 'convulsions', 'DiseaseOrPhenotypicFeature'],
 [15523499, 'beta-adrenergic receptors', 'GeneOrGeneProduct'],
 [24132704, 'corn oil', 'ChemicalEntity'],
 [20105310, 'asp receptor', 'GeneOrGeneProduct'],
 [28260056, 'threonine protein kinase b', 'GeneOrGeneProduct'],
 [17221831, 'purine nucleoside phosphorylase', 'GeneOrGeneProduct'],
 [15630069, 'diabetes', 'DiseaseOrPhenotypicFeature'],
 [15200408, 'up', 'GeneOrGeneProduct'],
 [17035713, 'tumor', 'DiseaseOrPhenotypicFeature'],
 [18674790, 'rats', 'OrganismTaxon'],
 [16369751, 'malignancies', 'DiseaseOrPhenotypicFeature'],
 [27999109, 'insulitis', 'DiseaseOrPhenotypicFeature'],
 [27248656, 'tnf-alpha', 'GeneOrGeneProduct'],
 [24743235, 'csf-1r', 'GeneOrGeneProduct'],
 [15241482, '